# CASMI 2026 — prize-path inference v2: own spectrum→fingerprint transformer + PubChem candidate channel

Everything here is trained by us on the competition `train.parquet` (kernel `dalloliogm/casmi26-own-fpnet-train`) and scored against a candidate pool of training structures ∪ COCONUT (CC BY 4.0). No third-party model weights.

Candidates: pool window (train ∪ COCONUT) **plus** the most-documented PubChem compounds in the same window (own rebuild of NCBI Extras, `dalloliogm/casmi26-pubchem-own-tier`); tier-only candidates get a tunable penalty and a popularity (log1p substances + PubMed ids) term, tuned on held-out identities whose answer is removed from the pool.

Per molecule: neutral mass → ±10 ppm pool window → score `F·z` (z = mean per-spectrum fingerprint logits, z-scored within the window) + a thresholded library-cosine bonus for strong spectral matches → top 25 unique structures.

In [ ]:
import os, re, sys, glob, time, math, subprocess, importlib.metadata, hashlib
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc, scipy.sparse as sp, torch, torch.nn as nn, torch.nn.functional as F
try: importlib.metadata.version("rdkit")
except importlib.metadata.PackageNotFoundError:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"; w = sorted(glob.glob(f"/kaggle/input/**/rdkit-*-{tag}-*.whl", recursive=True)); assert w
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", w[-1]])
def find(name, env):
    p = os.environ.get(env) or next(iter(glob.glob(f"/kaggle/input/**/{name}", recursive=True)), None); assert p, name; return p
TRAIN = find("train.parquet", "CASMI_TRAIN"); DATA_DIR = os.path.dirname(TRAIN)
POOL_NPZ = find("pool.npz", "POOL_NPZ")
_ck = glob.glob("/kaggle/input/**/ownfp_best.pt", recursive=True) + glob.glob("/kaggle/input/**/ownfp.pt", recursive=True)
_ck = sorted(_ck, key=lambda p: ("train3" not in p, "best" not in p, "train2" not in p, p))      # prefer the regularised run (train3 best EMA)
CKPT = os.environ.get("OWN_CKPT") or (_ck[0] if _ck else None); assert CKPT, "ownfp.pt not found"
OUT_DIR = os.environ.get("CASMI_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
dev = "cuda" if torch.cuda.is_available() else "cpu"; MAXP = 128; PPM = 10.0
ADDUCTS = ["[M+H]+","[M+NH4]+","[M-H2O+H]+","[M-2H2O+H]+","[M+Na]+","[M+K]+","[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+Cl]-"]
print(TRAIN, POOL_NPZ, CKPT, dev)

In [ ]:
class SinEmb(nn.Module):
    def __init__(s, dim, lo=-2.0, hi=3.3):
        super().__init__(); n = dim // 2
        s.register_buffer("inv", 2 * math.pi / torch.pow(10.0, (hi - lo) * torch.linspace(0, 1, n) + lo))
    def forward(s, x): a = x.unsqueeze(-1) * s.inv; return torch.cat([a.sin(), a.cos()], -1)
class Block(nn.Module):
    def __init__(s, d, h, p):
        super().__init__(); s.h = h; s.n1 = nn.LayerNorm(d); s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d); s.n2 = nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(p), nn.Linear(4 * d, d)); s.dr = nn.Dropout(p)
    def forward(s, x, pad):
        B, N, Dm = x.shape; q, k, v = s.qkv(s.n1(x)).view(B, N, 3, s.h, Dm // s.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=(~pad)[:, None, None, :])
        x = x + s.dr(s.o(a.transpose(1, 2).reshape(B, N, Dm))); return x + s.dr(s.ff(s.n2(x)))
class SpecNet(nn.Module):
    def __init__(s, nbits, d=384, layers=6, heads=6, p=0.1):
        super().__init__(); s.nbits = nbits
        s.mz = SinEmb(d); s.nl = SinEmb(d); s.inten = nn.Linear(2, d); s.pk = nn.Linear(3 * d, d)
        s.prec = SinEmb(d); s.ad = nn.Embedding(len(ADDUCTS) + 1, d); s.ce = SinEmb(d // 4, 0.0, 2.2); s.gl = nn.Linear(d + d // 4 + 4, d)
        s.blocks = nn.ModuleList([Block(d, heads, p) for _ in range(layers)]); s.norm = nn.LayerNorm(d)
        s.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(p), nn.Linear(2048, nbits))
    def forward(s, mz, it, pad, prec, ad, ce, ce_known, merged, mode):
        B, N = mz.shape; nl = (prec[:, None] - mz).clamp(min=0)
        pk = s.pk(torch.cat([s.mz(mz), s.nl(nl), s.inten(torch.stack([it, it * it], -1))], -1))
        g = s.gl(torch.cat([s.prec(prec), s.ce(ce / 10.0 + 1.0), ce_known[:, None], merged[:, None], mode[:, None], torch.log1p(prec)[:, None] / 10.0], -1)) + s.ad(ad)
        x = torch.cat([g[:, None], pk], 1); pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in s.blocks: x = b(x, pad)
        x = s.norm(x); m = (~pad[:, 1:]).float()[..., None]; mean = (x[:, 1:] * m).sum(1) / m.sum(1).clamp(min=1)
        return s.head(torch.cat([x[:, 0], mean], -1))

def batch_tensors(S, idx, aug=False):
    n = S["nv"][idx].astype(np.int64); mz = torch.from_numpy(S["mz"][idx]).to(dev); it = torch.from_numpy(S["it"][idx].astype(np.float32)).to(dev)
    pad = torch.arange(MAXP, device=dev)[None] >= torch.from_numpy(n).to(dev)[:, None]
    if aug:  # peak dropout + small m/z jitter
        drop = (torch.rand_like(it) < 0.1) & ~pad; it = it.masked_fill(drop, 0.0); mz = mz + torch.randn_like(mz) * 0.002
    T = lambda a, dt=torch.float32: torch.from_numpy(np.asarray(a)).to(dev, dt)
    return (mz, it, pad, T(S["prec"][idx]), T(S["adduct"][idx].astype(np.int64), torch.long), T(S["ce"][idx]), T(S["ce_known"][idx]),
            T(S["n_merged"][idx] / 4.0), T(S["mode"][idx]))

def unpack(packed, nb):
    sh = torch.arange(7, -1, -1, device=packed.device, dtype=torch.uint8)
    return ((packed.unsqueeze(-1) >> sh) & 1).reshape(*packed.shape[:-1], -1)[..., :nb].float()



In [ ]:
def prep_peaks(mz, it, prec):
    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)
    if len(mz) == 0: return None
    k = (mz > 0) & (mz <= prec + 2.0); mz, it = mz[k], it[k]
    if len(mz) == 0 or it.max() <= 0: return None
    it = it / it.max(); k = it >= 1e-3; mz, it = mz[k], it[k]
    if len(mz) > MAXP: s = np.argsort(-it)[:MAXP]; mz, it = mz[s], it[s]
    o = np.argsort(mz); return mz[o].astype(np.float32), np.sqrt(it[o]).astype(np.float16)



In [ ]:
ck = torch.load(CKPT, map_location=dev); ck["model"] = ck["model"] if "mrr" in ck else ck.get("ema", ck["model"]); pool = np.load(POOL_NPZ, allow_pickle=True)
pkeys, psmi, pmass, pfp_np = pool["keys"], pool["smiles"], pool["mass"], pool["fp"]; nb = ck["nbits"]; assert nb == len(pool["bits"])
pfp = torch.from_numpy(pfp_np).to(dev); net = SpecNet(nb, ck["d"], ck["layers"], max(1, ck["d"] // 64)).to(dev); net.load_state_dict(ck["model"]); net.eval()
use_amp = dev == "cuda"; print(len(pkeys), "pool structures; step", ck["step"])
PROTON, E = 1.007276467, 0.000548580
H2O, NA, K, CL, CH2O2, NH4 = 18.0105646863, 22.9897692809, 38.96370668, 34.96885268, 46.00547930326, 18.0343740
DELTA = {"[M+H]+": PROTON, "[M+NH4]+": NH4 - E, "[M-H2O+H]+": PROTON - H2O, "[M-2H2O+H]+": PROTON - 2*H2O, "[M+Na]+": NA - E, "[M+K]+": K - E,
         "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O, "[M+CH2O2-H]-": CH2O2 - PROTON, "[M+Cl]-": CL + E}
def rows_to_S(df):
    n = len(df); mz = np.zeros((n, MAXP), np.float32); it = np.zeros((n, MAXP), np.float16); nv = np.zeros(n, np.int16); ok = np.zeros(n, bool)
    for i, r in enumerate(df.itertuples()):
        pk = prep_peaks(r.ms2_mzs, r.ms2_normalized_intensities, r.precursor_mz)
        if pk is None: continue
        k = len(pk[0]); mz[i, :k] = pk[0]; it[i, :k] = pk[1]; nv[i] = k; ok[i] = True
    ce = [(np.mean([x for x in c if x == x]) if c is not None and len([x for x in c if x == x]) else 0.0) for c in df.collision_energy_ev]
    ck_ = np.array([1.0 if (c is not None and len([x for x in c if x == x])) else 0.0 for c in df.collision_energy_ev], np.float32)
    nm = np.array([min(len([x for x in c if x == x]), 8) if (c is not None and len([x for x in c if x == x])) else 1 for c in df.collision_energy_ev], np.int16)
    S = dict(mz=mz, it=it, nv=nv, prec=df.precursor_mz.to_numpy(np.float32), adduct=np.array([ADDUCTS.index(a) if a in ADDUCTS else len(ADDUCTS) for a in df.adduct], np.int16),
             mode=np.where(df.ionization_mode == "positive", 1.0, -1.0).astype(np.float32), ce=np.array(ce, np.float32), ce_known=ck_, n_merged=nm)
    return S, ok
@torch.no_grad()
def mol_z(g):
    S, ok = rows_to_S(g); ix = np.flatnonzero(ok)
    if len(ix) == 0: return None
    with torch.autocast(dev, dtype=torch.float16, enabled=use_amp): z = net(*batch_tensors(S, ix)).float()
    return z.mean(0)
def neutral_mass(g):
    d = g.adduct.map(DELTA); return float((g.precursor_mz - d).median())

In [ ]:
# ---- PubChem own tier + fingerprint helpers ----
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
RDLogger.DisableLog("rdApp.*")
TIER = os.path.dirname(find("pc_off.npy", "TIER_OFF"))
pc_mass = np.load(f"{TIER}/pc_mass.npy", mmap_mode="r"); pc_off = np.load(f"{TIER}/pc_off.npy", mmap_mode="r"); pc_buf = np.load(f"{TIER}/pc_smiles.npy", mmap_mode="r")
pc_lsid = np.load(f"{TIER}/pc_lsid.npy", mmap_mode="r"); pc_lpmid = np.load(f"{TIER}/pc_lpmid.npy", mmap_mode="r"); print("tier rows", len(pc_mass))
BITS = pool["bits"]; _G = {}
def _gens():
    if not _G:
        g = rdFingerprintGenerator
        _G.update(ecfp4=g.GetMorganGenerator(radius=2, fpSize=2048), ecfp6=g.GetMorganGenerator(radius=3, fpSize=2048),
                  fcfp4=g.GetMorganGenerator(radius=2, fpSize=1024, atomInvariantsGenerator=g.GetMorganFeatureAtomInvGen()),
                  ap=g.GetAtomPairGenerator(fpSize=1024), tt=g.GetTopologicalTorsionGenerator(fpSize=1024))
    return _G
def cand_info(smi):
    """-> (inchikey14, packed selected fingerprint) or None."""
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        g = _gens(); parts = [g[k].GetFingerprintAsNumPy(m).astype(np.uint8) for k in ("ecfp4", "ecfp6", "fcfp4", "ap", "tt")]
        parts.append(np.array(list(MACCSkeys.GenMACCSKeys(m)), np.uint8)); raw = np.concatenate(parts)
        return Chem.MolToInchiKey(m)[:14], np.packbits(raw[BITS])
    except Exception: return None
from multiprocessing import Pool as MPool
def tier_candidates(mass, n_top=2000):
    tol = mass * PPM * 1e-6; a, b = np.searchsorted(pc_mass, [mass - tol, mass + tol])
    if b <= a: return []
    p = pc_lsid[a:b].astype(np.float32) + pc_lpmid[a:b].astype(np.float32); top = np.argsort(-p)[:n_top]
    return [(bytes(pc_buf[pc_off[a + j]:pc_off[a + j + 1]]).decode("ascii"), float(p[j])) for j in top]


In [ ]:
# ---- library vectors (strong-match bonus) ----
BIN, NBIN = 0.02, int(1300 / 0.02) + 1
def vec(mzs, its, prec):
    r, c, v = [], [], []; keep = np.zeros(len(mzs), bool)
    for i, (mz, it, p) in enumerate(zip(mzs, its, prec)):
        mz = np.asarray(mz, float); it = np.asarray(it, float)
        if len(mz) == 0 or it.max() <= 0: continue
        it = it / it.max(); m = (mz <= p + 2) & (it >= 0.01) & (mz < 1300) & (mz > 0); mz, it = mz[m], it[m]
        if len(mz) < 2: continue
        if len(mz) > 100: t = np.argsort(-it)[:100]; mz, it = mz[t], it[t]
        b = (mz / BIN).astype(np.int64); s = pd.Series(np.sqrt(it)).groupby(b).max(); x = s.to_numpy(); x = x / np.linalg.norm(x)
        r.append(np.full(len(x), i)); c.append(s.index.to_numpy()); v.append(x); keep[i] = True
    if not r: return sp.csr_matrix((len(mzs), NBIN)), keep
    return sp.csr_matrix((np.concatenate(v), (np.concatenate(r), np.concatenate(c))), shape=(len(mzs), NBIN)), keep
kidx = {k: i for i, k in enumerate(pkeys)}
def library_for(keys, exclude_keys=frozenset(), exclude_rows=frozenset()):
    ks = pa.array(sorted(set(keys) - set(exclude_keys))); mats, metas, off = [], [], 0
    for b in pq.ParquetFile(TRAIN).iter_batches(batch_size=50000, columns=["inchikey14", "ms2_mzs", "ms2_normalized_intensities", "precursor_mz"]):
        t = pa.Table.from_batches([b]); t = t.append_column("row", pa.array(np.arange(off, off + t.num_rows))); off += t.num_rows
        t = t.filter(pc.is_in(t["inchikey14"], value_set=ks))
        if t.num_rows == 0: continue
        d = t.to_pandas()
        if exclude_rows: d = d[~d.row.isin(exclude_rows)]
        X, kp = vec(d.ms2_mzs, d.ms2_normalized_intensities, d.precursor_mz.to_numpy(float)); mats.append(X[np.flatnonzero(kp)]); metas.append(d.loc[kp, ["inchikey14"]])
    X = sp.vstack(mats).tocsr() if mats else sp.csr_matrix((0, NBIN)); meta = pd.concat(metas, ignore_index=True) if metas else pd.DataFrame({"inchikey14": []})
    return X, meta.groupby("inchikey14").indices
SMI = {}
def components(df, exclude_keys=frozenset(), exclude_rows=frozenset(), use_tier=True, lib_exclude_keys=frozenset()):
    mm = df.groupby("molecule_id").apply(neutral_mass); win = {}
    for m, x in mm.items():
        if np.isfinite(x): win[m] = (x, ) + tuple(np.searchsorted(pmass, [x * (1 - PPM * 1e-6), x * (1 + PPM * 1e-6)]))
    allk = {pkeys[i] for _, lo, hi in win.values() for i in range(lo, min(hi, lo + 20000))} - set(exclude_keys)
    LX, lib_idx = library_for(allk - set(lib_exclude_keys), set(exclude_keys) | set(lib_exclude_keys), exclude_rows); out = {}; mp = MPool(os.cpu_count())
    for mol, g in df.groupby("molecule_id"):
        if mol not in win: out[mol] = None; continue
        x, lo, hi = win[mol]; hi = min(hi, lo + 20000); z = mol_z(g)
        if z is None: out[mol] = None; continue
        keys0 = list(pkeys[lo:hi]); ok = np.array([k not in exclude_keys for k in keys0], bool)
        fp_all = (unpack(pfp[lo:hi], nb) @ z).cpu().numpy() if hi > lo else np.zeros(0, np.float32)
        keys = [k for k, o in zip(keys0, ok) if o]; fp = list(fp_all[ok]); smiles = [s_ for s_, o in zip(psmi[lo:hi], ok) if o]
        pop = [0.0] * len(keys); tier = [False] * len(keys); pos = {k: i for i, k in enumerate(keys)}
        if use_tier:
            tc = tier_candidates(x); res = mp.map(cand_info, [t[0] for t in tc], chunksize=64); seen = set()
            F_new, k_new, s_new, p_new = [], [], [], []
            for (smi, p_), r in zip(tc, res):
                if r is None: continue
                k, pk = r
                if k in pos: pop[pos[k]] = max(pop[pos[k]], p_); continue          # documented pool structure: attach popularity
                if k in seen: continue
                seen.add(k); k_new.append(k); s_new.append(smi); p_new.append(p_); F_new.append(pk)
            if F_new:
                Fm = unpack(torch.from_numpy(np.stack(F_new)).to(dev), nb); fn = (Fm @ z).cpu().numpy()
                keys += k_new; smiles += s_new; fp += list(fn); pop += p_new; tier += [True] * len(k_new)
        keys = np.array(keys, dtype=object); fp = np.array(fp, np.float32); lib = np.zeros(len(keys), np.float32)
        have = [j for j, k in enumerate(keys) if k in lib_idx]
        if have:
            Q, kp = vec(g.ms2_mzs, g.ms2_normalized_intensities, g.precursor_mz.to_numpy(float)); Q = Q[np.flatnonzero(kp)]
            if Q.shape[0]:
                rows = np.concatenate([lib_idx[keys[j]] for j in have]); own = np.concatenate([[q] * len(lib_idx[keys[j]]) for q, j in enumerate(have)])
                Sm = (Q @ LX[rows].T).toarray()
                for q, j in enumerate(have): lib[j] = Sm[:, own == q].max(1).mean()
        SMI.update(dict(zip(keys, smiles)))
        out[mol] = dict(keys=keys, fp=fp, lib=lib, pop=np.array(pop, np.float32), tier=np.array(tier, np.float32))
    mp.close(); return out
def blend(c, w):
    fz = (c["fp"] - c["fp"].mean()) / (c["fp"].std() + 1e-6); t = w["t"]
    pz = (c["pop"] - c["pop"].mean()) / (c["pop"].std() + 1e-6)
    return fz + w["lib"] * np.maximum(c["lib"] - t, 0) / (1 - t) + w["pop"] * pz - w["delta"] * c["tier"]
def rank(comps, w, k=25):
    return {m: ([] if c is None else [c["keys"][j] for j in np.argsort(-blend(c, w))[:k]]) for m, c in comps.items()}
def mrr(r, truth): return float(np.mean([(1 / (r[m].index(t) + 1)) if t in r.get(m, []) else 0.0 for m, t in truth.items()]))

## Validation (identity-disjoint hold-out = structures with md5(inchikey14)%100 < 3, never trained on)

In [ ]:
N_VAL = int(os.environ.get("N_VAL", 300)); rng = np.random.RandomState(1)
meta = pq.read_table(TRAIN, columns=["inchikey14", "adduct"]).to_pandas(); meta["row"] = np.arange(len(meta)); meta = meta[meta.adduct.isin(DELTA)]
isval = lambda k: int(hashlib.md5(k.encode()).hexdigest()[:8], 16) % 100 < 3
vk = np.array(sorted(k for k in meta.inchikey14.unique() if isval(k) and k in kidx)); v2 = rng.choice(vk, min(N_VAL, len(vk)), replace=False) if len(vk) else vk
cnt = meta[~meta.inchikey14.map(isval)].groupby("inchikey14").size(); v1 = rng.choice(cnt[cnt >= 2].index.to_numpy(), min(N_VAL // 2, int((cnt >= 2).sum())), replace=False)
rows2 = meta[meta.inchikey14.isin(set(v2))]; rows1 = meta[meta.inchikey14.isin(set(v1))].groupby("inchikey14", group_keys=False).apply(lambda g: g.sample(frac=0.5, random_state=0) if len(g) > 2 else g.iloc[:1])
want = np.array(sorted(set(rows2.row) | set(rows1.row))); parts, off = [], 0
for b in pq.ParquetFile(TRAIN).iter_batches(batch_size=50000, columns=["inchikey14", "ms2_mzs", "ms2_normalized_intensities", "adduct", "ionization_mode", "precursor_mz", "collision_energy_ev"]):
    lo, hi = np.searchsorted(want, [off, off + b.num_rows])
    if hi > lo: d = pa.Table.from_batches([b]).take(pa.array(want[lo:hi] - off)).to_pandas(); d["row"] = want[lo:hi]; parts.append(d)
    off += b.num_rows
val_df = pd.concat(parts, ignore_index=True); val_df["molecule_id"] = val_df.inchikey14; truth = {k: k for k in val_df.inchikey14.unique()}
cls2 = {k for k in truth if k in set(v2)}; clsA = {k for k in truth if k in set(v2[::2])}; clsB = {k for k in truth if k in set(v2[1::2])}; t0 = time.time()
v2a, v2b = set(v2[::2]), set(v2[1::2])   # A: answer stays in the pool (library spectra removed); B: removed from library AND pool (only PubChem can supply it)
vc = components(val_df, exclude_keys=v2b, lib_exclude_keys=v2a | v2b, exclude_rows=set(rows1.row)); print("components", f"{time.time()-t0:.0f}s")
def grp(w):
    r = rank(vc, w); return mrr(r, {m: truth[m] for m in clsA}), mrr(r, {m: truth[m] for m in clsB}), mrr(r, {m: t for m, t in truth.items() if m not in cls2})
import itertools
res = []
for t_, wl, wp, dl in itertools.product((0.85,), (0, 5), (0, 0.1, 0.25, 0.5), (0, 1, 2, 4, 99)):
    w = dict(t=t_, lib=wl, pop=wp, delta=dl); a_, b_, c_ = grp(w); res.append((w, a_, b_, c_))
for w, a_, b_, c_ in res: print(f"lib={w['lib']} pop={w['pop']} delta={w['delta']}: A(in pool) {a_:.3f} B(pubchem-only) {b_:.3f} class1 {c_:.3f}")
best = max(res, key=lambda x: 0.6 * x[1] + 0.1 * x[2] + 0.3 * x[3]); W = best[0]; print("chosen", W, best[1:])


## Predict test

In [ ]:
test = pq.read_table(f"{DATA_DIR}/test.parquet").to_pandas(); tc = components(test); r = rank(tc, W)
smi = SMI; rows = [(m, ";".join(smi[k] for k in r.get(m, [])[:25]) or "CCO") for m in test.molecule_id.unique()]
sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"]); sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv"); assert set(sub.molecule_id) == set(ss.molecule_id) and sub.molecule_id.is_unique and sub.notna().all().all()
print("submission OK", sub.shape, (sub.smiles == "CCO").sum(), "without candidates")